# Notebook 6: Geospatial Launch Site Visualization with Folium
## IBM Data Science Professional Certificate Capstone Project

### Learning Objectives:
1. Understand latitude and longitude geographic coordinates.
2. Build interactive maps using `folium`.
3. Add launch site markers, safety perimeter circles, and coordinate popups.
4. Cluster individual launch outcomes using `MarkerCluster`.
5. Compute geodesic distances from launch sites to coastlines, railways, highways, and cities using the Haversine formula.

---
### 1. What is Folium?
Folium is a powerful Python library that bridges Python data structures with Leaflet.js, generating interactive, shareable HTML maps.

In [ ]:
# Step 1: Import Folium and dependencies
import folium
from folium.plugins import MarkerCluster, MousePosition
import pandas as pd
import math
import os

df = pd.read_csv("../data/processed/dataset_part_2.csv")
print("Folium imported successfully!")

### 2. Define Launch Site Coordinates and Distance Calculator

In [ ]:
# Launch Site Coordinates
LAUNCH_SITES = {
    "CCAFS SLC 40": {"lat": 28.562302, "lon": -80.577356, "coastline": (28.56367, -80.56763)},
    "KSC LC 39A": {"lat": 28.573255, "lon": -80.646895, "coastline": (28.58022, -80.64242)},
    "VAFB SLC 4E": {"lat": 34.632834, "lon": -120.610745, "coastline": (34.63467, -120.62534)}
}

def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0  # Earth radius in km
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2.0)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2.0)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

print("Distance calculator initialized.")

### 3. Create the Interactive Launch Site Map

In [ ]:
# Step 3: Initialize Folium Map centered on the US
site_map = folium.Map(location=[30.0, -95.0], zoom_start=4, tiles="OpenStreetMap")

# Add Launch Site Markers and Circles
for site, info in LAUNCH_SITES.items():
    folium.Circle(
        location=[info["lat"], info["lon"]],
        radius=1000,
        color="#1A237E",
        fill=True,
        fill_color="#3F51B5",
        fill_opacity=0.3,
        tooltip=f"{site} Safety Buffer"
    ).add_to(site_map)
    
    folium.Marker(
        location=[info["lat"], info["lon"]],
        icon=folium.Icon(color="blue", icon="rocket", prefix="fa"),
        popup=f"<b>{site}</b>",
        tooltip=site
    ).add_to(site_map)

# Add MarkerCluster for Individual Launches
cluster = MarkerCluster(name="Falcon 9 Launches").add_to(site_map)
for _, r in df.iterrows():
    site = r["LaunchSite"]
    coords = LAUNCH_SITES.get(site, {"lat": 28.56, "lon": -80.57})
    is_success = r["Class"] == 1
    folium.Marker(
        location=[coords["lat"], coords["lon"]],
        icon=folium.Icon(color="green" if is_success else "red", icon="check" if is_success else "times", prefix="fa"),
        tooltip=f"Flight #{r['FlightNumber']}: {'Success' if is_success else 'Failure'}"
    ).add_to(cluster)

# Save map
os.makedirs("../visuals/folium", exist_ok=True)
map_out = "../visuals/folium/launch_sites_map.html"
site_map.save(map_out)
print(f"Map successfully saved to: {map_out}")

---
### Practice Exercise & Check for Understanding:
1. Why are all launch sites located along the coastlines of Florida and California? *(Answer: Rockets launching to the east or south fly over open ocean, so any launch failure drops safely into water rather than populated land).*
2. What is the approximate distance from Cape Canaveral SLC 40 to the nearest coastline? *(Answer: Approximately 1.0 to 1.2 kilometers).*

### Summary & Next Steps:
We generated the interactive geographic launch map. Next, in Notebook 7, we train, tune, and evaluate our Machine Learning models to predict landing success!